<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/12.Model_Pruning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 12 — Model Pruning



GPT-2 versi terkecil menyimpan sekitar 124 juta bobot, dan banyak di antaranya bernilai sangat dekat dengan nol. Ide **pruning** (pemangkasan) berangkat dari pengamatan itu: jika sebuah bobot hampir nol, menjadikannya tepat nol seharusnya hampir tidak mengubah perilaku model. Imbalannya diharapkan berupa model yang lebih kecil atau lebih cepat.

Kata kuncinya adalah "diharapkan". Bab ini mengujinya langsung pada GPT-2 lewat tiga pertanyaan:

1. Berapa banyak bobot yang bisa dibuang sebelum kualitas model runtuh?
2. Apakah bobot yang menjadi nol otomatis membuat file lebih kecil dan inference lebih cepat?
3. Jika model rusak setelah dipangkas, bagaimana menyembuhkannya tanpa menumbuhkan kembali bobot yang sudah dibuang?

Listing dari buku ikut diuji. Listing yang tidak bisa berjalan apa adanya, atau berjalan tetapi salah sasaran pada GPT-2, ditampilkan sebagai kotak kutipan pendek dan langsung diikuti versi yang bekerja.



## Persiapan: pastikan GPU benar-benar bekerja

Sel berikut tidak berhenti pada pertanyaan "apakah ada GPU?". Ia mengalokasikan matriks 2048×2048 di GPU, mengalikannya, menunggu GPU selesai (`torch.cuda.synchronize`), lalu memastikan hasilnya finite. Jika salah satu langkah gagal, notebook berhenti dengan instruksi Colab; tidak ada peralihan diam-diam ke CPU.

Semua model memakai **float32**. Model terbesar di bab ini hanya sekitar 0,5 GB, jadi memori T4 bukan kendala, dan float32 membuat selisih loss yang kecil antar-eksperimen tidak tercampur noise presisi rendah. T4 juga tidak punya dukungan bfloat16 native, jadi resep "pakai bf16" tidak cocok untuknya.

Mode CPU hanya untuk uji kecil dan harus diminta eksplisit dengan `os.environ["CH12_DEVICE"] = "cpu"` sebelum sel ini; profilnya diperkecil dan angkanya tidak setara run GPU. Satu bagian nanti (quantization dinamis) memang wajib berjalan di CPU, dan bagian itu diberi label sendiri.

In [1]:
import os
import sys
import importlib.util
import subprocess

if importlib.util.find_spec("datasets") is None:      # Colab biasanya sudah punya; instal hanya jika belum ada
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import torch

RUN_DEVICE = os.environ.get("CH12_DEVICE", "cuda").strip().lower()
DTYPE = torch.float32


def cuda_gate():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA tidak tersedia. Di Colab: Runtime > Change runtime type > pilih GPU (mis. T4), "
            "lalu jalankan ulang dari sel pertama. Uji kecil di CPU hanya lewat opt-in eksplisit: "
            "os.environ['CH12_DEVICE'] = 'cpu' sebelum sel ini.")
    device = torch.device("cuda")
    a = torch.randn(2048, 2048, device=device, dtype=DTYPE)
    b = a @ a.T
    torch.cuda.synchronize()
    if not torch.isfinite(b).all().item():
        raise RuntimeError("Uji matmul di GPU menghasilkan NaN/Inf; ganti runtime sebelum melanjutkan.")
    del a, b
    torch.cuda.empty_cache()
    props = torch.cuda.get_device_properties(device)
    free, total = torch.cuda.mem_get_info()
    print(f"GPU OK: {props.name} | compute capability {props.major}.{props.minor} | "
          f"memori bebas {free / 1e9:.1f} / {total / 1e9:.1f} GB | bf16 native: {props.major >= 8}")
    return device


if RUN_DEVICE == "cuda":
    DEVICE = cuda_gate()
elif RUN_DEVICE == "cpu":
    DEVICE = torch.device("cpu")
    print("MODE CPU (opt-in eksplisit): profil diperkecil, angka tidak setara run GPU.")
else:
    raise ValueError(f"CH12_DEVICE={RUN_DEVICE!r} tidak didukung; pakai 'cuda' (default) atau 'cpu'.")

PROFILE = {
    "cuda": dict(val_seqs=32, eval_batch=8, train_seqs=256, sweep=(0.1, 0.3, 0.5, 0.7, 0.9),
                 mini_pretrain_steps=400, mini_heal_steps=60, kd_steps=60, kd_batch=4,
                 mlp_tokens=(16, 128), timing_reps=50),
    "cpu": dict(val_seqs=4, eval_batch=2, train_seqs=16, sweep=(0.1, 0.5, 0.9),
                mini_pretrain_steps=60, mini_heal_steps=10, kd_steps=4, kd_batch=2,
                mlp_tokens=(4, 128), timing_reps=5),
}[DEVICE.type]
torch.manual_seed(0)
print("device:", DEVICE, "| dtype:", DTYPE)
print("profil:", PROFILE)

GPU OK: Tesla T4 | compute capability 7.5 | memori bebas 15.5 / 15.6 GB | bf16 native: False
device: cuda | dtype: torch.float32
profil: {'val_seqs': 32, 'eval_batch': 8, 'train_seqs': 256, 'sweep': (0.1, 0.3, 0.5, 0.7, 0.9), 'mini_pretrain_steps': 400, 'mini_heal_steps': 60, 'kd_steps': 60, 'kd_batch': 4, 'mlp_tokens': (16, 128), 'timing_reps': 50}


## GPT-2, penggaris validasi, dan satu fakta arsitektur

Sebelum memangkas, kita butuh penggaris. Penggarisnya adalah **loss validasi**: rata-rata cross-entropy per token (satuan *nats*) pada potongan teks WikiText-2, masing-masing 128 token. Loss lebih mudah dibayangkan lewat **perplexity** = exp(loss): perplexity 40 berarti model rata-rata sebingung orang yang menebak dari 40 token yang sama-sama mungkin. Semakin kecil, semakin baik.

Sel ini memuat GPT-2 124M pretrained ke GPU dan menyimpan salinan bobot awalnya di RAM CPU (bukan VRAM), supaya setiap eksperimen bisa mulai dari titik yang sama lewat `restore()`. Tokenisasi teks berjalan di CPU karena itu pekerjaan I/O biasa; hanya tensor token hasilnya yang dipindah ke GPU. Terakhir, sel ini menghitung jenis layer di dalam GPT-2.

In [2]:
import math
import time
import copy
import gc
import io
import collections

import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
import transformers
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel
from transformers.pytorch_utils import Conv1D
from datasets import load_dataset

transformers.logging.set_verbosity_error()
print("torch", torch.__version__, "| transformers", transformers.__version__)

MODEL_ID = "gpt2"
SEQ_LEN = 128
tok = AutoTokenizer.from_pretrained(MODEL_ID)
gpt2 = GPT2LMHeadModel.from_pretrained(MODEL_ID).to(device=DEVICE, dtype=DTYPE).eval()
SD0 = {k: v.detach().to("cpu", copy=True) for k, v in gpt2.state_dict().items()}   # snapshot di RAM CPU

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")


def pack(split, n_seqs, seq_len=SEQ_LEN):
    """Tokenisasi baris demi baris (CPU), sambung, lalu potong menjadi n_seqs x seq_len token."""
    ids = []
    for text in split["text"]:
        if text.strip():
            ids.extend(tok(text)["input_ids"])
        if len(ids) >= n_seqs * seq_len:
            break
    assert len(ids) >= n_seqs * seq_len, "teks tidak cukup untuk profil ini"
    return torch.tensor(ids[: n_seqs * seq_len]).view(n_seqs, seq_len)


val_ids = pack(wiki["validation"], PROFILE["val_seqs"]).to(DEVICE)
train_ids = pack(wiki["train"], PROFILE["train_seqs"]).to(DEVICE)


@torch.no_grad()
def eval_loss(model, data=None, batch=None):
    """Rata-rata loss per token. Data dan model harus sudah berada di device yang sama."""
    data = val_ids if data is None else data
    batch = PROFILE["eval_batch"] if batch is None else batch
    model_device = next(model.parameters()).device
    assert data.device == model_device, f"data di {data.device}, model di {model_device}"
    assert len(data) % batch == 0
    model.eval()
    losses = [model(input_ids=data[i:i + batch], labels=data[i:i + batch], use_cache=False).loss.item()
              for i in range(0, len(data), batch)]
    loss = float(np.mean(losses))
    assert math.isfinite(loss), "loss tidak finite"
    return loss


def conv_modules(model):
    return [m for m in model.modules() if isinstance(m, Conv1D)]


def remove_all_pruning(model):
    """Patenkan semua mask yang masih aktif (weight_orig * weight_mask -> weight biasa)."""
    for module in model.modules():
        for hook in list(module._forward_pre_hooks.values()):
            if isinstance(hook, prune.BasePruningMethod):
                prune.remove(module, hook._tensor_name)


def restore(model=gpt2):
    remove_all_pruning(model)
    model.load_state_dict(SD0)
    model.eval()


def global_sparsity(model):
    assert not any(n.endswith("_orig") for n, _ in model.named_parameters()), "panggil prune.remove dulu"
    total = zeros = 0
    for p in model.parameters():          # parameter unik: wte dan lm_head yang tied dihitung sekali
        total += p.numel()
        zeros += (p == 0).sum().item()
    return zeros / total


linear_names = [n for n, m in gpt2.named_modules() if isinstance(m, nn.Linear)]
conv_names = [n for n, m in gpt2.named_modules() if isinstance(m, Conv1D)]
N_PARAMS = sum(p.numel() for p in gpt2.parameters())
TIED = gpt2.lm_head.weight is gpt2.transformer.wte.weight
c_fc0 = gpt2.transformer.h[0].mlp.c_fc
print(f"parameter unik: {N_PARAMS / 1e6:.1f} jt | nn.Linear: {linear_names} | Conv1D: {len(conv_names)} modul")
print(f"lm_head tied ke wte: {TIED} | bentuk wte: {tuple(gpt2.transformer.wte.weight.shape)}")
print(f"contoh Conv1D h.0.mlp.c_fc: weight {tuple(c_fc0.weight.shape)} = (input, output)")
assert linear_names == ["lm_head"] and len(conv_names) == 4 * gpt2.config.n_layer and TIED, \
    "narasi bab ini mengasumsikan GPT-2 di transformers memakai Conv1D dan weight tying"

SP0 = global_sparsity(gpt2)
BASE_LOSS = eval_loss(gpt2)
print(f"sparsity awal: {SP0:.2%} | val loss awal: {BASE_LOSS:.3f} nats | perplexity {math.exp(BASE_LOSS):.1f} | "
      f"{len(val_ids)} sekuens x {SEQ_LEN} token di {val_ids.device}")

torch 2.11.0+cu130 | transformers 5.20.2


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

parameter unik: 124.4 jt | nn.Linear: ['lm_head'] | Conv1D: 48 modul
lm_head tied ke wte: True | bentuk wte: (50257, 768)
contoh Conv1D h.0.mlp.c_fc: weight (768, 3072) = (input, output)
sparsity awal: 0.00% | val loss awal: 4.109 nats | perplexity 60.9 | 32 sekuens x 128 token di cuda:0


Hasil sensus di atas menentukan nasib listing pertama buku, jadi perhatikan dua hal:

- **Hanya ada satu `nn.Linear`, yaitu `lm_head`.** Proyeksi attention dan MLP GPT-2 (`c_attn`, `c_proj`, `c_fc`) memakai `Conv1D`: 4 modul per blok × 12 blok = 48. `Conv1D` menyimpan bobot berbentuk *(input, output)*, kebalikan dari `nn.Linear` yang *(output, input)*.
- **`lm_head` adalah tensor yang sama dengan embedding input `wte`** (*weight tying*). Matriks 50.257×768 itu dipakai dua kali: untuk membaca token yang masuk dan untuk memberi skor token berikutnya. Mengubah salah satunya berarti mengubah keduanya.

## Magnitude pruning: lihat dulu pada matriks kecil

Cara pemangkasan paling sederhana adalah **magnitude pruning**: di setiap matriks, bobot dengan nilai mutlak terkecil dijadikan nol. Di PyTorch fungsinya `prune.l1_unstructured(module, name="weight", amount=0.3)`. Disebut *unstructured* karena bobot dibuang satu per satu di posisi mana pun, dan `amount=0.3` berarti 30% bobot dengan |w| terkecil.

Sebelum menyentuh GPT-2, kita lihat apa yang sebenarnya dilakukan PyTorch pada matriks 2×4 dengan `amount=0.5`.

In [3]:
toy = nn.Linear(4, 2, bias=False).to(device=DEVICE, dtype=DTYPE)
with torch.no_grad():
    toy.weight.copy_(torch.tensor([[0.90, -0.10, 0.40, -0.05],
                                   [0.02, -0.70, 0.30, 0.60]]))

prune.l1_unstructured(toy, name="weight", amount=0.5)
print("parameter :", [n for n, _ in toy.named_parameters()])
print("buffer    :", [n for n, _ in toy.named_buffers()])
print("weight_mask:\n", toy.weight_mask.cpu())
print("weight = weight_orig * weight_mask:\n", toy.weight.detach().cpu())
assert int(toy.weight_mask.sum()) == 4

prune.remove(toy, "weight")
print("setelah prune.remove -> parameter:", [n for n, _ in toy.named_parameters()],
      "| buffer:", [n for n, _ in toy.named_buffers()])

parameter : ['weight_orig']
buffer    : ['weight_mask']
weight_mask:
 tensor([[1., 0., 1., 0.],
        [0., 1., 0., 1.]])
weight = weight_orig * weight_mask:
 tensor([[ 0.9000, -0.0000,  0.4000, -0.0000],
        [ 0.0000, -0.7000,  0.0000,  0.6000]])
setelah prune.remove -> parameter: ['weight'] | buffer: []


Empat bobot dengan |w| terkecil (0,02; −0,05; −0,1; 0,3) menjadi nol, empat sisanya utuh. Yang lebih penting adalah *cara* PyTorch melakukannya. Ia tidak menimpa bobot asli:

- bobot asli dipindah ke parameter `weight_orig` dan tetap bisa dilatih;
- pola 0/1 disimpan di buffer `weight_mask`;
- sebelum setiap forward, `weight` dihitung ulang sebagai `weight_orig * weight_mask`.

Mekanisme ini disebut **reparametrisasi**. `prune.remove` mengakhirinya: hasil perkalian ditulis permanen ke `weight` biasa, sedangkan `weight_orig` dan `weight_mask` dibuang. Kedua istilah ini akan muncul lagi saat kita melatih model yang sudah dipangkas.

Sekarang listing pertama buku.

> **Listing buku: placeholder dan sasaran yang meleset**
>
> ```python
> model = ...  # Load or define your LLM model
>
> # Prune 30% of the lowest magnitude weights in all Linear layers
> for name, module in model.named_modules():
>     if isinstance(module, torch.nn.Linear):
>         prune.l1_unstructured(module, name='weight', amount=0.3)
> ```
>
> Dijalankan apa adanya, `...` adalah objek `Ellipsis` yang sah di Python, sehingga run historis berhenti dengan `AttributeError: 'ellipsis' object has no attribute 'named_modules'`. Masalah yang lebih penting baru terlihat setelah placeholder diisi GPT-2 sungguhan; sel berikut menjalankan loop yang sama dan mengukur akibatnya.

In [4]:
restore()
pruned_names = []
for name, module in gpt2.named_modules():
    if isinstance(module, torch.nn.Linear):          # sasaran versi buku
        prune.l1_unstructured(module, name="weight", amount=0.3)
        pruned_names.append(name)
for name in pruned_names:
    prune.remove(gpt2.get_submodule(name), "weight")

wte = gpt2.transformer.wte.weight
sp = global_sparsity(gpt2)
book_loss = eval_loss(gpt2)
print(f"modul yang dipangkas       : {pruned_names}")
print(f"Conv1D yang ikut dipangkas : {sum(n in conv_names for n in pruned_names)} dari {len(conv_names)}")
print(f"sparsity global            : {sp:.1%} (sebelumnya {SP0:.1%}; target listing: 30%)")
print(f"entri wte yang kini nol    : {(wte == 0).float().mean().item():.1%} | "
      f"lm_head masih tied: {gpt2.lm_head.weight is wte}")
print(f"val loss                   : {BASE_LOSS:.3f} -> {book_loss:.3f}")
assert abs((sp - SP0) - 0.3 * wte.numel() / N_PARAMS) < 1e-4    # nol baru = tepat 30% dari wte
restore()

modul yang dipangkas       : ['lm_head']
Conv1D yang ikut dipangkas : 0 dari 48
sparsity global            : 9.3% (sebelumnya 0.0%; target listing: 30%)
entri wte yang kini nol    : 30.0% | lm_head masih tied: True
val loss                   : 4.109 -> 4.120


Tiga hal terjadi sekaligus:

1. **Sparsity global hanya 9,3%, bukan 30%.** Satu-satunya `nn.Linear` adalah `lm_head`; 30% dari 38,6 juta bobotnya setara 9,3% dari 124,4 juta parameter unik. Ke-48 `Conv1D`, tempat sebagian besar komputasi transformer berlangsung, tidak tersentuh.
2. **Embedding input ikut berlubang.** Karena `lm_head` dan `wte` berbagi tensor, `prune.remove` menulis nol ke tensor bersama itu. Kini 30% entri embedding bernilai nol, artinya model membaca token lewat tabel yang bolong.
3. **Loss naik** dari nilai awal. Run historis mencatat 3,762 → 3,793. Kerusakannya kecil, tetapi datang dari bagian yang justru paling tidak ingin kita pangkas.

Sasaran yang benar untuk GPT-2 adalah `Conv1D`, sementara pasangan `wte`/`lm_head` dibiarkan utuh.

## Under the hood: seberapa jauh kita bisa memangkas?

Dengan sasaran yang benar, pertanyaan utama bab ini bisa dijawab. Kita pangkas ke-48 `Conv1D` dengan `amount` 10% sampai 90% dan mengukur loss validasi setiap kali, selalu mulai dari bobot asli.

Sparsity global akan lebih kecil dari `amount`. Bobot `Conv1D` hanya sekitar 68% dari seluruh parameter; sisanya embedding token dan posisi yang tidak kita sentuh. Memangkas 90% Conv1D berarti kira-kira 0,9 × 68% ≈ 61% nol secara global.

In [5]:
CONV_WEIGHTS = sum(m.weight.numel() for m in conv_modules(gpt2))
print(f"bobot Conv1D: {CONV_WEIGHTS / 1e6:.1f} jt dari {N_PARAMS / 1e6:.1f} jt ({CONV_WEIGHTS / N_PARAMS:.0%})\n")
print(f"{'amount':>7} {'sparsity global':>16} {'val loss':>9} {'Δ loss':>8} {'perplexity':>11}")
sweep_rows = []
for amount in (0.0,) + PROFILE["sweep"]:
    restore()
    for module in conv_modules(gpt2) if amount > 0 else []:
        prune.l1_unstructured(module, name="weight", amount=amount)
        prune.remove(module, "weight")
    loss = BASE_LOSS if amount == 0 else eval_loss(gpt2)
    sp = global_sparsity(gpt2)
    sweep_rows.append((amount, sp, loss))
    print(f"{amount:>7.0%} {sp:>16.1%} {loss:>9.3f} {loss - BASE_LOSS:>+8.3f} {math.exp(loss):>11.1f}")
restore()

bobot Conv1D: 84.9 jt dari 124.4 jt (68%)

 amount  sparsity global  val loss   Δ loss  perplexity
     0%             0.0%     4.109   +0.000        60.9
    10%             6.8%     4.127   +0.018        62.0
    30%            20.5%     4.443   +0.334        85.1
    50%            34.1%     6.473   +2.364       647.5
    70%            47.8%     8.372   +4.263      4323.5
    90%            61.4%     9.819   +5.710     18382.0


Bacalah kolom **Δ loss** dan **perplexity** bersama-sama. Selisih loss 0,02 nats berarti perplexity naik sekitar 2% (exp(0,02) ≈ 1,02), hampir tak terasa. Selisih 1 nats berarti perplexity naik hampir tiga kali lipat (exp(1) ≈ 2,7).

Run historis memberi gambaran bentuk kurvanya: pada 10% loss hanya naik 0,02 nats (perplexity 43 → 44); pada 30% perplexity menjadi 65; pada 50% melonjak ke 526; dan pada 90% mencapai sekitar 12.700, alias model hampir menebak acak. Pola **datar lalu tebing** inilah yang perlu Anda cari di output sendiri.

Maknanya punya dua sisi. Redundansi di transformer pretrained memang nyata: sebagian kecil bobot terkecil bisa dibuang hampir gratis, dan itulah sebabnya magnitude pruning sesederhana ini masih dipakai. Tetapi redundansi itu terbatas. Klaim "dampak minimal pada akurasi" hanya berlaku di sisi kiri kurva; setelah melewati titik patah, model butuh penyembuhan (fine-tune) atau kriteria pemangkasan yang lebih cermat.

## Under the hood: mengapa API `prune` berbentuk aneh?

Buku menjelaskan cara kerja `prune.l1_unstructured` dengan menulis versinya sendiri. Baris pertamanya sudah bermasalah:

> **Listing buku: fungsi dengan nama bertitik**
>
> ```python
> def prune.l1_unstructured(module, name, amount):
>     """Prunes weights with lowest L1 norm magnitude in a module's tensor"""
>     tensor = getattr(module, name)
>     n_params_to_prune = int(amount * tensor.numel())
>     ...
> ```
>
> Nama fungsi tidak boleh mengandung titik, jadi Python menolaknya dengan `SyntaxError` sebelum apa pun berjalan. Di bawah, fungsi ini diberi nama sah, `book_l1_unstructured`, dengan isi mengikuti buku: hitung ambang, buat mask, ganti `weight` dengan `nn.Parameter` baru, lalu pasang *hook* yang mengalikan bobot dengan mask sebelum setiap forward.

Kesalahan sintaks itu luka yang dangkal. Pertanyaan yang lebih penting: **apakah model masih bisa dilatih setelah dipangkas dengan cara buku?** Supaya jawabannya keluar dalam hitungan detik, kita memakai GPT-2 mini (2 layer, dimensi 64) yang dilatih pada korpus proxy: teks WikiText-2 yang sama, tetapi kosakatanya diringkas ke 4.096 token tersering. Persiapan korpus (tokenisasi dan pemetaan token) adalah pekerjaan CPU; blok token hasilnya dipindah ke GPU.

In [6]:
MINI_SEQ, MINI_VOCAB = 64, 4096
texts = [t for t in wiki["train"][:10000]["text"] if t.strip()]
raw = []
for ids in tok(texts)["input_ids"]:                  # tahap CPU: tokenisasi + pemetaan kosakata
    raw.extend(ids + [tok.eos_token_id])
remap = {t: i + 1 for i, (t, _) in enumerate(collections.Counter(raw).most_common(MINI_VOCAB - 1))}
mini_ids = torch.tensor([remap.get(t, 0) for t in raw])          # 0 = token di luar 4.095 tersering
blocks = mini_ids[: len(mini_ids) // MINI_SEQ * MINI_SEQ].view(-1, MINI_SEQ)
assert len(blocks) >= 2248, "korpus proxy terlalu kecil"
MINI_TRAIN = blocks[:2000].to(DEVICE)
MINI_EVAL = blocks[2200:2248].to(DEVICE)
print(f"korpus proxy: {len(raw):,} token -> {len(blocks):,} blok x {MINI_SEQ} | "
      f"train {tuple(MINI_TRAIN.shape)}, eval {tuple(MINI_EVAL.shape)} di {MINI_TRAIN.device}")


def mini_gpt2(seed=42):
    torch.manual_seed(seed)
    cfg = GPT2Config(vocab_size=MINI_VOCAB, n_positions=MINI_SEQ, n_embd=64, n_layer=2, n_head=2,
                     resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, bos_token_id=0, eos_token_id=0)
    return GPT2LMHeadModel(cfg).to(device=DEVICE, dtype=DTYPE)


def mini_batch(step, bs=8):
    start = (step * bs) % (len(MINI_TRAIN) - bs)
    return MINI_TRAIN[start:start + bs]

korpus proxy: 649,057 token -> 10,141 blok x 64 | train (2000, 64), eval (48, 64) di cuda:0


Eksperimennya dirancang agar tidak ada tempat bersembunyi. Kedua versi memangkas 50% bobot setiap `Conv1D`, dan optimizer **hanya** memegang bobot-bobot terpangkas itu. Jika loss turun, penurunannya pasti berasal dari bobot tersebut. Selain loss, sel ini mencatat total perubahan bobot dan apakah parameter yang dipegang optimizer menerima gradien sama sekali.

In [7]:
def book_l1_unstructured(module, name, amount):     # nama sah; isi mengikuti listing buku
    tensor = getattr(module, name)
    n_params_to_prune = int(amount * tensor.numel())
    threshold = torch.kthvalue(tensor.abs().view(-1), n_params_to_prune).values
    mask = tensor.abs() > threshold
    pruned_tensor = tensor.clone() * mask
    setattr(module, name, torch.nn.Parameter(pruned_tensor))
    module.register_buffer(f"{name}_mask", mask)
    module.register_forward_pre_hook(
        lambda m, _: setattr(m, name, torch.nn.Parameter(getattr(m, name) * getattr(m, f"{name}_mask"))))
    return mask


for label, use_book in (("reimplementasi buku", True), ("torch.nn.utils.prune", False)):
    m = mini_gpt2()
    targets = conv_modules(m)
    if use_book:
        for t in targets:
            book_l1_unstructured(t, "weight", 0.5)
        live = [t.weight for t in targets]            # Parameter yang diserahkan ke optimizer
    else:
        for t in targets:
            prune.l1_unstructured(t, "weight", amount=0.5)
        live = [t.weight_orig for t in targets]       # reparametrisasi: inilah yang dilatih
    before = [p.detach().clone() for p in live]
    opt = torch.optim.AdamW(live, lr=5e-3)            # optimizer HANYA memegang bobot terpangkas
    m.train()
    losses = []
    for step in range(25):
        x = mini_batch(step)
        loss = m(input_ids=x, labels=x, use_cache=False).loss
        loss.backward()
        if step == 0:
            got_grad = all(p.grad is not None for p in live)
        opt.step()
        opt.zero_grad(set_to_none=True)
        losses.append(loss.detach())
    losses = torch.stack(losses).cpu()
    assert torch.isfinite(losses).all()
    delta = sum((p.detach() - b).abs().sum().item() for p, b in zip(live, before))
    print(f"{label:<21}: loss {losses[0]:.3f} -> {losses[-1]:.3f} | perubahan bobot (L1) {delta:9.2f} | "
          f"parameter optimizer menerima gradien: {got_grad}")
    if use_book:
        assert delta == 0.0 and not got_grad
    else:
        assert delta > 0 and got_grad
        assert all((t.weight[t.weight_mask == 0] == 0).all() for t in targets)
    del m, opt, live, before

reimplementasi buku  : loss 8.266 -> 8.264 | perubahan bobot (L1)      0.00 | parameter optimizer menerima gradien: False
torch.nn.utils.prune : loss 8.266 -> 8.038 | perubahan bobot (L1)   1274.08 | parameter optimizer menerima gradien: True


Versi buku **membeku**: parameter milik optimizer tidak pernah menerima gradien, sehingga total perubahan bobotnya tepat 0. Penyebabnya ada di hook. Setiap forward membuat `nn.Parameter` *baru*; gradien mengalir ke objek baru itu, sedangkan objek yang dipegang optimizer tidak pernah lagi dipakai dalam forward. Loss yang sedikit bergeser hanyalah efek batch yang berbeda-beda. Run historis menunjukkan polanya: 8,266 → 8,264 untuk versi buku, 8,266 → 8,038 untuk `torch.nn.utils.prune`.

`torch.nn.utils.prune` menghindari masalah ini lewat reparametrisasi yang kita lihat pada matriks kecil. Optimizer memegang `weight_orig`, `weight` dihitung darinya, jadi gradien sampai ke parameter yang benar. Gradien di posisi yang di-mask bernilai nol, sehingga bobot terpangkas tetap nol sepanjang training; sel tadi memeriksanya dengan `assert`. Bentuk API yang terasa janggal (`weight_orig`, `weight_mask`, lalu `remove()`) memang dirancang untuk alasan ini.

## Structured vs unstructured pruning

Nol hasil unstructured pruning tersebar acak di dalam matriks. Bentuk matriksnya tidak berubah, dan perkalian matriks padat (*dense*) tetap mengalikan nol-nol itu. **Structured pruning** membuang satu struktur utuh, misalnya neuron, head attention, atau layer, sehingga matriksnya bisa benar-benar diperkecil.

Di MLP GPT-2, satu "neuron" adalah satu dari 3.072 unit tersembunyi di `c_fc`. Buku memakai `ln_structured` dengan `dim=0` dan menyebutnya membuang baris, yaitu neuron:

> **Listing buku: sumbu yang benar untuk `nn.Linear`, salah untuk `Conv1D`**
>
> ```python
> prune.ln_structured(module, name='weight', amount=0.3, n=2, dim=0)
> ```
>
> Baris ini berjalan tanpa error. Pada `nn.Linear` (bobot *output × input*), `dim=0` memang memilih neuron output. Pada `Conv1D` GPT-2 (bobot *input × output*), sumbu yang sama memilih fitur input. Karena perbedaan inilah intinya, sel berikut sengaja menjalankan keduanya sebagai pembanding.

In [8]:
restore()
c_fc = gpt2.transformer.h[0].mlp.c_fc
n_in, n_out = c_fc.weight.shape
print(f"h.0.mlp.c_fc: {type(c_fc).__name__}, weight {tuple(c_fc.weight.shape)} -> "
      f"baris = {n_in} fitur input, kolom = {n_out} neuron")
for dim, label in ((0, "dim=0 (listing buku)"), (1, "dim=1")):
    prune.ln_structured(c_fc, name="weight", amount=0.3, n=2, dim=dim)
    w = c_fc.weight
    zero_rows = int((w.abs().sum(dim=1) == 0).sum())
    zero_cols = int((w.abs().sum(dim=0) == 0).sum())
    print(f"{label:<21}: baris input nol {zero_rows}/{n_in} | kolom neuron nol {zero_cols}/{n_out}")
    expected = (round(0.3 * n_in), 0) if dim == 0 else (0, round(0.3 * n_out))
    assert (zero_rows, zero_cols) == expected
    restore()

h.0.mlp.c_fc: Conv1D, weight (768, 3072) -> baris = 768 fitur input, kolom = 3072 neuron
dim=0 (listing buku) : baris input nol 230/768 | kolom neuron nol 0/3072
dim=1                : baris input nol 0/768 | kolom neuron nol 922/3072


`dim=0` menolkan 230 dari 768 baris, yaitu fitur input yang masuk ke *semua* neuron, tanpa membuang satu neuron pun. `dim=1` menolkan 922 dari 3.072 kolom, yaitu neuron yang sesungguhnya. Kodenya sama, maknanya berbeda, dan tidak ada error yang memberi peringatan.

Masih ada satu detail. `ln_structured` hanya memasang mask pada `weight`. Neuron yang kolom bobotnya nol tetap mengirim GELU(bias)-nya ke `c_proj`, jadi ia belum benar-benar hilang. Untuk menghapus neuron sepenuhnya, bias neuron itu juga harus di-mask; fungsi `prune_mlp_neurons` di bawah melakukannya dengan `prune.custom_from_mask`.

Sekarang perbandingan yang adil: 12 matriks `c_fc` yang sama, 30% bobotnya menjadi nol, dengan tiga cara memilih bobot mana yang dibuang.

In [9]:
def prune_mlp_neurons(mlp, amount, with_bias=True):
    """Structured pruning neuron MLP GPT-2: kolom c_fc (dim=1), opsional ikut bias neuronnya."""
    prune.ln_structured(mlp.c_fc, name="weight", amount=amount, n=2, dim=1)
    keep = mlp.c_fc.weight_mask[0].clone()          # 1 = neuron dipertahankan
    assert (mlp.c_fc.weight_mask == keep).all()      # mask berbentuk kolom utuh
    if with_bias:
        prune.custom_from_mask(mlp.c_fc, name="bias", mask=keep)
    return keep


ARMS = (("unstructured L1", "u"),
        ("structured, bias neuron tetap hidup", "s_weight"),
        ("structured, neuron dihapus penuh", "s_full"))
struct_loss = {}
for label, kind in ARMS:
    restore()
    for block in gpt2.transformer.h:
        if kind == "u":
            prune.l1_unstructured(block.mlp.c_fc, name="weight", amount=0.3)
        else:
            prune_mlp_neurons(block.mlp, 0.3, with_bias=(kind == "s_full"))
    w_sparsity = np.mean([(b.mlp.c_fc.weight == 0).float().mean().item() for b in gpt2.transformer.h])
    struct_loss[kind] = eval_loss(gpt2)
    print(f"{label:<36}: sparsity bobot c_fc {w_sparsity:.1%} | val loss {struct_loss[kind]:.3f} "
          f"({struct_loss[kind] - BASE_LOSS:+.3f} dari awal)")
restore()

unstructured L1                     : sparsity bobot c_fc 30.0% | val loss 4.285 (+0.176 dari awal)
structured, bias neuron tetap hidup : sparsity bobot c_fc 30.0% | val loss 9.050 (+4.941 dari awal)
structured, neuron dihapus penuh    : sparsity bobot c_fc 30.0% | val loss 6.877 (+2.768 dari awal)


Pada sparsity bobot yang praktis sama, structured pruning jauh lebih mahal. Run historis mencatat unstructured 3,980 dan structured dengan bias tetap hidup 8,771, dari baseline 3,762: selisih sekitar 4,8 nats. Alasannya geometris. Unstructured bebas memilih 30% bobot terlemah di mana pun, sedangkan structured dipaksa membuang neuron utuh, termasuk bobot-bobot kuat yang kebetulan berada di neuron bernorma kecil.

Baris kedua dan ketiga hanya berbeda pada bias neuron. Perbandingan ini tidak ada di run historis, jadi bacalah langsung dari output Anda: selisihnya menunjukkan seberapa banyak informasi yang masih dibawa neuron "terpangkas" lewat biasnya.

Kalau structured semahal itu, mengapa tetap dipakai? Karena hanya nol berbentuk kolom yang bisa dihapus dari matriks. Klaim itu kita ukur berikutnya.

### Under the hood: nol yang tersebar vs kolom yang dihapus

Kita ambil satu blok MLP dan membuat empat varian:

- **dense asli**: `c_fc` 768×3072;
- **30% nol tersebar**: unstructured, bentuk matriks tetap 768×3072;
- **mask neuron aktif**: structured 30% (weight + bias) yang masih berupa reparametrisasi, sehingga ada perkalian `weight_orig * weight_mask` di setiap forward;
- **slim**: 922 neuron dihapus secara fisik, `c_fc` menjadi 768×2150 dan `c_proj` 2150×768.

Sebelum mengukur waktu, kita pastikan varian slim menghitung fungsi yang sama dengan varian mask neuron; tanpa itu, membandingkan kecepatannya tidak bermakna. Di GPU, waktu diukur dengan CUDA event: beberapa iterasi pemanasan, lalu median dari banyak pengulangan, dengan sinkronisasi setelah setiap pengukuran.

In [10]:
def slim_mlp(mlp, keep):
    """Bangun ulang blok MLP padat tanpa neuron yang tidak di-keep: bentuk matriks benar-benar mengecil."""
    idx = keep.nonzero().squeeze(1)
    n_embd = mlp.c_fc.weight.shape[0]
    slim = copy.deepcopy(mlp)                         # mlp padat tanpa mask aktif (deepcopy menolak mask aktif)
    slim.c_fc = Conv1D(len(idx), n_embd)              # Conv1D(nf=output, nx=input)
    slim.c_proj = Conv1D(n_embd, len(idx))
    with torch.no_grad():
        slim.c_fc.weight.copy_(mlp.c_fc.weight[:, idx])
        slim.c_fc.bias.copy_(mlp.c_fc.bias[idx])
        slim.c_proj.weight.copy_(mlp.c_proj.weight[idx, :])
        slim.c_proj.bias.copy_(mlp.c_proj.bias)
    return slim.to(device=DEVICE, dtype=DTYPE).eval()


@torch.no_grad()
def check_slim_equivalence(dense, masked, slim, keep, x):
    """(1) Struktur dicek bit demi bit; (2) output masked dan slim dibandingkan dengan referensi float64."""
    k = keep.bool()
    idx = k.nonzero().squeeze(1)
    w1, b1, w2, b2 = dense.c_fc.weight, dense.c_fc.bias, dense.c_proj.weight, dense.c_proj.bias
    assert torch.equal(slim.c_fc.weight, w1[:, idx]) and torch.equal(slim.c_fc.bias, b1[idx]), "kolom/bias c_fc slim salah"
    assert torch.equal(slim.c_proj.weight, w2[idx, :]) and torch.equal(slim.c_proj.bias, b2), "baris/bias c_proj slim salah"
    assert not masked.c_fc.weight[:, ~k].any() and not masked.c_fc.bias[~k].any(), "neuron terpangkas masih hidup"
    assert torch.equal(masked.c_fc.weight[:, k], w1[:, k]) and torch.equal(masked.c_fc.bias[k], b1[k])
    assert torch.equal(masked.c_proj.weight, w2) and torch.equal(masked.c_proj.bias, b2)

    y_mask, y_slim = masked(x), slim(x)
    assert torch.isfinite(y_mask).all() and torch.isfinite(y_slim).all(), "output NaN/Inf"
    x64, w1, b1, w2, b2 = (t.double() for t in (x, w1, b1, w2, b2))
    ref = (dense.act(x64 @ w1 + b1) * k) @ w2 + b2               # definisi "neuron dihapus", float64
    terms = ((x64.abs() @ w1.abs() + b1.abs()) * k) @ w2.abs() + b2.abs()   # skala: jumlah |suku| dua GEMM
    eps = torch.finfo(x.dtype).eps
    if x.dtype == torch.float32 and x.is_cuda and torch.get_float32_matmul_precision() != "highest":
        eps = 2.0 ** -10                                          # matmul boleh TF32 (mantissa 10 bit)
    tol = 4 * math.sqrt(w1.shape[0] + w2.shape[0]) * eps * terms  # batas pembulatan n suku: 4·√n·eps·Σ|suku|
    worst = max(((y.double() - ref).abs() / tol).max().item() for y in (y_mask, y_slim))
    assert worst <= 1.0, f"galat {worst:.2f}x batas pembulatan float: bukan sekadar urutan penjumlahan"
    return (y_mask - y_slim).abs().max().item(), worst


def sync():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


@torch.no_grad()
def median_ms(module, x, reps):
    for _ in range(5):                                # pemanasan: alokasi memori dan pemilihan kernel
        module(x)
    sync()
    times = []
    for _ in range(reps):
        if DEVICE.type == "cuda":
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            module(x)
            end.record()
            torch.cuda.synchronize()
            times.append(start.elapsed_time(end))
        else:
            t0 = time.perf_counter()
            module(x)
            times.append((time.perf_counter() - t0) * 1e3)
    return float(np.median(times))


restore()
dense = copy.deepcopy(gpt2.transformer.h[0].mlp).eval()
zeros30 = copy.deepcopy(dense)
prune.l1_unstructured(zeros30.c_fc, name="weight", amount=0.3)
prune.remove(zeros30.c_fc, "weight")
masked = copy.deepcopy(dense)
keep = prune_mlp_neurons(masked, 0.3, with_bias=True)
slim = slim_mlp(dense, keep)

x = torch.randn(*PROFILE["mlp_tokens"], dense.c_fc.weight.shape[0], device=DEVICE, dtype=DTYPE)
max_diff, worst = check_slim_equivalence(dense, masked, slim, keep, x)
print(f"neuron tersisa {int(keep.sum())}/{keep.numel()} | selisih maks output mask vs slim: {max_diff:.1e} | "
      f"galat vs referensi float64: {worst:.4f}x batas pembulatan | matmul fp32: {torch.get_float32_matmul_precision()}")

reps = PROFILE["timing_reps"]
clock = "CUDA event" if DEVICE.type == "cuda" else "perf_counter CPU"
print(f"input {tuple(x.shape)} | median {reps} pengulangan | {clock}")
t_dense = median_ms(dense, x, reps)
for label, module in (("dense asli", dense), ("30% nol tersebar", zeros30),
                      ("mask neuron aktif", masked), ("slim (kolom dihapus fisik)", slim)):
    ms = t_dense if module is dense else median_ms(module, x, reps)
    print(f"{label:<27}: c_fc {str(tuple(module.c_fc.weight.shape)):<12} {ms:8.3f} ms | {ms / t_dense:.2f}x dense")
print(f"rasio FLOPs slim/dense: {int(keep.sum()) / keep.numel():.2f}")
del dense, zeros30, masked, slim, x

neuron tersisa 2150/3072 | selisih maks output mask vs slim: 2.7e-04 | galat vs referensi float64: 0.0009x batas pembulatan | matmul fp32: highest
input (16, 128, 768) | median 50 pengulangan | CUDA event
dense asli                 : c_fc (768, 3072)     6.198 ms | 1.00x dense
30% nol tersebar           : c_fc (768, 3072)     6.090 ms | 0.98x dense
mask neuron aktif          : c_fc (768, 3072)     6.092 ms | 0.98x dense
slim (kolom dihapus fisik) : c_fc (768, 2150)     4.237 ms | 0.68x dense
rasio FLOPs slim/dense: 0.70


Cara membaca output ini:

- **Kesetaraan slim dan mask** dicek dua lapis oleh `check_slim_equivalence`. Pertama, struktur dibandingkan bit demi bit: kolom dan bias `c_fc` slim harus persis kolom neuron yang dipertahankan, baris `c_proj` harus persis baris neuron itu, dan pada varian mask neuron terpangkas harus nol di bobot maupun bias. Kesalahan slicing atau masking gagal di sini tanpa toleransi apa pun. Kedua, output float masked dan slim dibandingkan dengan referensi float64 dari rumus yang sama. GEMM float32 dengan lebar berbeda (3072 vs 2150 neuron) boleh memilih kernel dan urutan penjumlahan yang berbeda, jadi hasilnya tidak harus identik bit demi bit (lihat *Numerical accuracy* di dokumentasi PyTorch). Toleransinya adalah batas galat pembulatan probabilistik untuk menjumlahkan *n* suku (Higham & Mary, 2019): 4·√n·eps·Σ|suku|, dengan n = 768 + 3072 suku pada dua GEMM dan eps dari dtype. Batas ini ikut membesar bila nilai aktivasi besar, sedangkan ambang absolut 10⁻⁴ tidak. Bila `torch.get_float32_matmul_precision()` bukan `highest` (TF32 diizinkan), eps disesuaikan menjadi 2⁻¹⁰. Angka "galat vs referensi" jauh di bawah 1 berarti selisih mask vs slim hanya efek urutan penjumlahan float.
- **Dense vs 30% nol tersebar** seharusnya hampir sama cepat, karena kernel matriks padat tetap mengalikan nol.
- **Mask aktif** tidak pernah lebih hemat dari dense, dan bisa sedikit lebih lambat karena ada perkalian mask tambahan setiap forward. Reparametrisasi berguna selama training, bukan saat inference.
- **Slim** hanya mengerjakan 70% FLOPs (2150/3072). Rasio waktunya jarang tepat 0,70: biaya tetap seperti peluncuran kernel, GELU, dan bias menariknya ke atas, sedangkan efek cache dan pilihan kernel matriks bisa menggesernya ke arah mana pun.

Angka milidetiknya bergantung pada GPU dan ukuran input, dan belum pernah diukur di Colab saat notebook ini ditulis. Pelajaran yang tidak bergantung pada hardware: manfaat kecepatan structured pruning baru muncul setelah layer dibangun ulang dengan bentuk yang lebih kecil. Mask saja tidak pernah membuat inference lebih cepat.

## Iterative pruning

Memangkas banyak bobot sekaligus melukai model. Resep buku memotongnya bertahap: pangkas 10% setiap 10 epoch, latih di antaranya agar model pulih, dan panggil `prune.remove` di akhir setiap ronde. Urutan langkah terakhir itu perlu diuji dulu. Sel berikut menjalankan tiga ronde `amount=0.1` pada satu matriks `c_fc` GPT-2 dengan dua disiplin: `remove()` setiap ronde seperti buku, dan mask dibiarkan menumpuk lalu `remove()` sekali di akhir.

In [11]:
restore()
c_fc = gpt2.transformer.h[0].mlp.c_fc
print("disiplin buku: prune.remove() setiap ronde")
for r in range(1, 4):
    prune.l1_unstructured(c_fc, name="weight", amount=0.1)
    prune.remove(c_fc, "weight")
    s = (c_fc.weight == 0).float().mean().item()
    print(f"  ronde {r}: sparsity c_fc = {s:.1%}")
    assert abs(s - 0.1) < 1e-3
restore()

print("mask dibiarkan menumpuk, remove() sekali di akhir")
for r in range(1, 4):
    prune.l1_unstructured(c_fc, name="weight", amount=0.1)
    s = (c_fc.weight == 0).float().mean().item()
    print(f"  ronde {r}: sparsity c_fc = {s:.1%}   (1 - 0.9^{r} = {1 - 0.9 ** r:.1%})")
    assert abs(s - (1 - 0.9 ** r)) < 1e-3
print("hook pruning pada c_fc:", [type(h).__name__ for h in c_fc._forward_pre_hooks.values()])
restore()

disiplin buku: prune.remove() setiap ronde
  ronde 1: sparsity c_fc = 10.0%
  ronde 2: sparsity c_fc = 10.0%
  ronde 3: sparsity c_fc = 10.0%
mask dibiarkan menumpuk, remove() sekali di akhir
  ronde 1: sparsity c_fc = 10.0%   (1 - 0.9^1 = 10.0%)
  ronde 2: sparsity c_fc = 19.0%   (1 - 0.9^2 = 19.0%)
  ronde 3: sparsity c_fc = 27.1%   (1 - 0.9^3 = 27.1%)
hook pruning pada c_fc: ['PruningContainer']


Versi buku **macet di 10%**. Setelah `remove()`, nol hasil ronde pertama menjadi bobot biasa bernilai 0, dan 0 adalah magnitudo terkecil yang mungkin. Ronde berikutnya mencari "10% bobot terkecil" dan menemukan nol-nol yang sama. Loop iteratif itu sebenarnya hanya memangkas sekali.

Tanpa `remove()`, PyTorch menumpuk mask dalam `PruningContainer` dan menerapkan setiap `amount` pada bobot yang **masih tersisa**: 10% → 19% → 27,1%, mengikuti rumus 1 − 0,9 pangkat k. Akibat praktisnya, untuk mencapai 30% setelah tiga ronde, `amount` per ronde harus sekitar 1 − 0,7^(1/3) ≈ 11,2%, bukan 10%. Aturan urutannya sederhana: `remove()` dipanggil sekali, setelah seluruh jadwal pemangkasan dan penyembuhan selesai.

## Pruning selama training vs setelah training

Ada dua waktu untuk memangkas. **Selama training**, bobot dibuang bertahap sementara model terus menyesuaikan diri; hasilnya biasanya lebih baik, tetapi mahal. **Setelah training**, model jadi dipangkas sekali; murah, tetapi lukanya lebih dalam dan sering perlu fine-tune penyembuh.

Fungsi training di listing buku tidak bisa dipakai langsung pada model Hugging Face:

> **Listing buku: loss dihitung dari objek output**
>
> ```python
> criterion = torch.nn.CrossEntropyLoss()
> ...
> inputs, targets = batch
> outputs = model(inputs)
> loss = criterion(outputs, targets)
> loss.backward()
> ```
>
> `model(inputs)` pada model Hugging Face mengembalikan objek `CausalLMOutputWithCrossAttentions`, bukan tensor, sehingga run historis berhenti dengan `TypeError: cross_entropy_loss(): argument 'input' (position 1) must be Tensor`. Mengganti `outputs` dengan `outputs.logits` pun belum cukup: logits berbentuk *(batch, seq, vocab)* sementara target *(batch, seq)*, dan pada language model kausal logits di posisi t harus dibandingkan dengan token di posisi t+1.

Jalur yang benar adalah memberi `labels=input_ids`; model menggeser label di dalamnya. Sel berikut memeriksa klaim itu secara numerik pada GPT-2, lalu mendefinisikan `train_one_batch` versi yang bekerja.

In [12]:
x = train_ids[:2]
with torch.no_grad():
    out = gpt2(input_ids=x, labels=x, use_cache=False)
    logits = out.logits                                              # (batch, seq, vocab)
    manual = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1))
print(f"type(output) = {type(out).__name__} | logits {tuple(logits.shape)} vs target {tuple(x.shape)}")
print(f"loss bawaan HF {out.loss.item():.6f} | CE manual dengan geser satu posisi {manual.item():.6f}")
assert torch.allclose(out.loss, manual, atol=1e-4)
del out, logits


def train_one_batch(model, input_ids, optimizer):
    """Versi yang bekerja: loss dari labels (digeser di dalam model), lalu step dan zero_grad."""
    model.train()
    loss = model(input_ids=input_ids, labels=input_ids, use_cache=False).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)
    return loss.detach()

type(output) = CausalLMOutputWithCrossAttentions | logits (2, 128, 50257) vs target (2, 128)
loss bawaan HF 3.988878 | CE manual dengan geser satu posisi 3.988878


Kedua angka loss identik sampai beberapa digit desimal: `labels=input_ids` memang menghasilkan cross-entropy token berikutnya dengan pergeseran satu posisi. Fungsi `train_one_batch` sekaligus melengkapi langkah yang tidak ada di cuplikan listing tadi: `optimizer.step()` dan `optimizer.zero_grad()`.

Sekarang pertanyaan penyembuhan. Buku menyarankan fine-tune setelah pruning untuk memulihkan kualitas. Ada jebakan halus di sini: jika fine-tune dilakukan **setelah** `prune.remove`, nol-nol tadi sudah menjadi bobot biasa. Gradien bebas mengubahnya, dan bobot yang "dibuang" bisa tumbuh kembali.

Kita uji pada GPT-2 mini. Model dilatih dulu sebagai versi "pretrained", lalu 50% bobot `Conv1D`-nya dipangkas dan disembuhkan dengan dua urutan:

- **A:** prune → `remove()` → fine-tune;
- **B:** prune → fine-tune dengan mask terpasang → `remove()`.

In [13]:
def train_steps(model, steps, lr=1e-3, offset=0):
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = torch.stack([train_one_batch(model, mini_batch(offset + s), opt) for s in range(steps)])
    assert torch.isfinite(losses).all()
    return losses


def mini_eval(model):
    return eval_loss(model, MINI_EVAL, batch=8)


def conv_sparsity(model):
    convs = conv_modules(model)
    return sum(int((c.weight == 0).sum()) for c in convs) / sum(c.weight.numel() for c in convs)


PRE, HEAL = PROFILE["mini_pretrain_steps"], PROFILE["mini_heal_steps"]
mini = mini_gpt2()
train_steps(mini, PRE)
base_mini = mini_eval(mini)
sd_mini = {k: v.detach().clone() for k, v in mini.state_dict().items()}

# A: prune -> remove -> fine-tune
mini.load_state_dict(sd_mini)
for c in conv_modules(mini):
    prune.l1_unstructured(c, name="weight", amount=0.5)
    prune.remove(c, "weight")
pruned_mini = mini_eval(mini)
sp_a0 = conv_sparsity(mini)
train_steps(mini, HEAL, offset=PRE)
sp_a1 = conv_sparsity(mini)
heal_a = mini_eval(mini)

# B: prune -> fine-tune dengan mask terpasang -> remove
mini.load_state_dict(sd_mini)
for c in conv_modules(mini):
    prune.l1_unstructured(c, name="weight", amount=0.5)
train_steps(mini, HEAL, offset=PRE)
sp_b = conv_sparsity(mini)
heal_b = mini_eval(mini)
remove_all_pruning(mini)
sp_b_final = conv_sparsity(mini)

print(f"mini pretrained ({PRE} step): loss {base_mini:.3f} | setelah prune 50% Conv1D: {pruned_mini:.3f}")
print(f"A. remove lalu fine-tune {HEAL} step   : loss {heal_a:.3f} | sparsity Conv1D {sp_a0:.1%} -> {sp_a1:.1%}")
print(f"B. fine-tune bermask {HEAL} step, remove: loss {heal_b:.3f} | sparsity Conv1D {sp_b:.1%} -> "
      f"{sp_b_final:.1%} setelah remove")
assert abs(sp_a0 - 0.5) < 1e-3 and sp_a1 < 0.05
assert abs(sp_b - 0.5) < 1e-3 and abs(sp_b_final - 0.5) < 1e-3
del mini, sd_mini

mini pretrained (400 step): loss 5.568 | setelah prune 50% Conv1D: 5.558
A. remove lalu fine-tune 60 step   : loss 5.548 | sparsity Conv1D 50.0% -> 0.0%
B. fine-tune bermask 60 step, remove: loss 5.577 | sparsity Conv1D 50.0% -> 50.0% setelah remove


Kolom yang membawa pelajaran utama adalah **sparsity**. Pada urutan A, sparsity jatuh dari 50% ke hampir 0% hanya dalam beberapa step: optimizer mengisi kembali posisi yang kita kira sudah dibuang, dan hasil akhirnya model padat yang *pernah* dipangkas. Pada urutan B, mask menjaga nol tetap nol, dan sparsity masih tepat 50% setelah `remove()`. Kedua fakta ini dijaga dengan `assert`, jadi berlaku di run mana pun.

Kolom loss lebih bergantung pada seberapa terlatih model mini itu. Di run historis, model mini hanya dilatih 120 step dan pemangkasan 50% nyaris tidak berdampak (5,987 → 5,986); tidak ada luka yang perlu disembuhkan. Profil GPU revisi ini melatihnya 400 step, jadi bandingkan sendiri loss setelah prune dengan loss setelah urutan A dan B. Urutan yang layak dihafal: **prune → sembuhkan dengan mask terpasang → remove**.

## Pruning + quantization (bagian ini sengaja berjalan di CPU)

Buku menggabungkan pruning dengan **quantization**, yaitu menyimpan bobot sebagai integer 8-bit (int8) alih-alih float 32-bit. Listingnya memangkas, meng-quantize, lalu "mengukur" ukuran model:

> **Listing buku: ukuran model dari memori GPU**
>
> ```python
> quantized_model = quant.quantize_dynamic(model, {torch.nn.Linear}, dtype=torch.qint8)
>
> # Check the size reduction
> print("Original model size:", torch.cuda.memory_allocated())
> print("Quantized model size:", torch.cuda.memory_allocated())
> ```
>
> Run historis mencetak `0` dan `0`. `torch.cuda.memory_allocated()` membaca memori GPU, padahal model hasil quantization dinamis hidup di CPU. Kedua baris juga dipanggil setelah quantization, jadi di GPU pun keduanya akan mencetak angka yang sama. Ukuran model seharusnya diukur dari byte bobot yang disimpan.

**Mengapa CPU?** `quantize_dynamic` menghasilkan modul dengan kernel int8 khusus CPU (fbgemm/x86 atau qnnpack); modul itu tidak bisa dipindah ke CUDA dengan `.to("cuda")`. Karena itu sel berikut membuat **salinan CPU eksplisit** dari bobot awal, sementara model GPU utama tidak disentuh. Ukuran dihitung dengan menyerialisasi `state_dict` ke buffer di memori, dan loss dibandingkan CPU lawan CPU pada 8 sekuens validasi agar waktu CPU tetap singkat. Sel ini juga menghitung berapa byte yang dibutuhkan bila bobot `Conv1D` terpangkas disimpan dalam format sparse CSR.

In [14]:
CPU = torch.device("cpu")
quantize_dynamic = getattr(torch.ao.quantization, "quantize_dynamic", None)
val_cpu = val_ids[: min(8, len(val_ids))].cpu()


def serialized_mb(model):
    buf = io.BytesIO()
    torch.save(model.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6


def conv_csr_mb(model):
    total = 0
    for m in conv_modules(model):
        csr = m.weight.detach().to_sparse_csr()
        total += sum(t.numel() * t.element_size() for t in (csr.crow_indices(), csr.col_indices(), csr.values()))
    return total / 1e6


if quantize_dynamic is None:
    print("DILEWATI: torch.ao.quantization.quantize_dynamic tidak tersedia di torch", torch.__version__)
else:
    if torch.backends.quantized.engine == "none":         # sebagian build torch belum memilih engine int8
        supported = torch.backends.quantized.supported_engines
        torch.backends.quantized.engine = next(e for e in ("x86", "fbgemm", "qnnpack") if e in supported)
    print("bagian CPU | quantized engine:", torch.backends.quantized.engine)
    cpu_model = GPT2LMHeadModel(gpt2.config).to(CPU, dtype=torch.float32).eval()   # salinan CPU eksplisit
    cpu_model.load_state_dict(SD0)
    assert cpu_model.lm_head.weight is cpu_model.transformer.wte.weight
    rows = [("GPT-2 asli (fp32)", serialized_mb(cpu_model), eval_loss(cpu_model, val_cpu, batch=2))]

    q_only = quantize_dynamic(cpu_model, {nn.Linear}, dtype=torch.qint8)      # inplace=False: salinan baru
    quantized = [n for n, m in q_only.named_modules()
                 if type(m).__module__.startswith("torch.ao.nn.quantized") and not n.endswith("_packed_params")]
    rows.append(("quantize {nn.Linear} saja", serialized_mb(q_only), eval_loss(q_only, val_cpu, batch=2)))
    del q_only

    for m in conv_modules(cpu_model):
        prune.l1_unstructured(m, name="weight", amount=0.4)
        prune.remove(m, "weight")
    rows.append(("pruned 40% Conv1D (dense)", serialized_mb(cpu_model), eval_loss(cpu_model, val_cpu, batch=2)))
    conv_dense = sum(m.weight.numel() * m.weight.element_size() for m in conv_modules(cpu_model)) / 1e6
    conv_csr = conv_csr_mb(cpu_model)

    pq = quantize_dynamic(cpu_model, {nn.Linear}, dtype=torch.qint8)
    rows.append(("pruned 40% + quantize", serialized_mb(pq), eval_loss(pq, val_cpu, batch=2)))
    assert all(p.device.type == "cpu" for p in pq.parameters())

    print(f"modul yang ter-quantize oleh {{nn.Linear}}: {quantized}")
    print(f"{'varian (CPU, ' + str(len(val_cpu)) + ' sekuens)':<29} {'ukuran':>10} {'val loss':>9}")
    for label, mb, loss in rows:
        print(f"{label:<29} {mb:>7.1f} MB {loss:>9.3f}")
    print(f"bobot Conv1D terpangkas 40%: format padat {conv_dense:.1f} MB | format CSR {conv_csr:.1f} MB")
    print("model GPU utama tetap di:", next(gpt2.parameters()).device)
    assert abs(rows[2][1] - rows[0][1]) < 0.1, "pruning dense seharusnya tidak mengubah ukuran file"
    del cpu_model, pq
    gc.collect()

bagian CPU | quantized engine: x86


/tmp/ipykernel_1397/1217557768.py:32: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  q_only = quantize_dynamic(cpu_model, {nn.Linear}, dtype=torch.qint8)      # inplace=False: salinan baru
/tmp/ipykernel_1397/1217557768.py:15: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request 

modul yang ter-quantize oleh {nn.Linear}: ['lm_head']
varian (CPU, 8 sekuens)           ukuran  val loss
GPT-2 asli (fp32)               497.8 MB     3.762
quantize {nn.Linear} saja       536.4 MB     8.995
pruned 40% Conv1D (dense)       497.8 MB     5.067
pruned 40% + quantize           536.4 MB     9.646
bobot Conv1D terpangkas 40%: format padat 339.7 MB | format CSR 612.0 MB
model GPU utama tetap di: cuda:0


Ada empat hal yang terbaca dari tabel ini.

**Pruning tidak mengecilkan file.** Ukuran model terpangkas 40% praktis sama dengan aslinya (selisihnya di bawah 0,1 MB, dijaga `assert`). Nol tetaplah angka float32 yang menempati 4 byte.

**Format sparse juga tidak otomatis menolong.** CSR menyimpan setiap bobot bukan-nol beserta indeks kolomnya: 4 byte nilai + 8 byte indeks = 12 byte, dibanding 4 byte di format padat. CSR baru lebih hemat bila lebih dari dua pertiga bobot bernilai nol; pada sparsity 40% ukurannya justru hampir dua kali lipat.

**Quantization `{nn.Linear}` membuat file membesar.** Hanya `lm_head` yang ter-quantize. Modul int8-nya memegang salinan sendiri (50.257 × 768 byte ≈ 38,6 MB), sementara `wte` float32 tetap wajib ada. Ikatan weight tying putus dan matriks raksasa itu tersimpan dua kali; run historis mencatat 497,8 → 536,4 MB.

**Loss dirusak oleh quantization, bahkan tanpa pruning.** Run historis mencatat loss asli 3,762, quantization saja 8,995, pruning 40% saja 5,067, dan gabungan keduanya 9,646. Quantization int8 per-tensor pada matriks embedding/head yang penuh nilai ekstrem memang resep kerusakan. Quantization yang lebih tepat (per-channel, weight-only, dan mencakup ke-48 `Conv1D`) adalah topik bab berikutnya di buku, *Quantization*.

## Pruning + knowledge distillation

Kombinasi kedua di buku adalah **knowledge distillation**: model kecil (*student*, hasil pemangkasan) dilatih meniru distribusi output model besar (*teacher*). Loop distilasi buku:

> **Listing buku: loop distilasi (format diringkas)**
>
> ```python
> teacher_model = ...  # Larger, fully trained model
> student_model = ...  # Smaller model to be distilled and pruned
>
> def distillation_loss(student_outputs, teacher_outputs, temperature):
>     return torch.nn.KLDivLoss()(
>         torch.nn.functional.log_softmax(student_outputs / temperature),
>         torch.nn.functional.softmax(teacher_outputs / temperature))
> ```
>
> Placeholder `...` kembali membuat run historis berhenti (`TypeError: 'ellipsis' object is not callable`). Di baliknya ada empat masalah yang tidak memunculkan error: loop tidak pernah memanggil `optimizer.zero_grad()`; `softmax`/`log_softmax` tanpa `dim=`; `KLDivLoss()` dengan reduksi default `'mean'` yang merata-rata atas semua elemen *batch × seq × vocab*, bukan per token; dan tidak ada faktor T².

Hilangnya `zero_grad` paling berbahaya sekaligus paling mudah dibuktikan: jalankan `backward()` dua kali pada batch yang sama tanpa mengosongkan gradien di antaranya.

In [15]:
probe = mini_gpt2()
probe.train()
x = mini_batch(0)
probe(input_ids=x, labels=x, use_cache=False).loss.backward()
g1 = torch.cat([p.grad.flatten() for p in probe.parameters()]).clone()
probe(input_ids=x, labels=x, use_cache=False).loss.backward()         # tanpa zero_grad di antaranya
g2 = torch.cat([p.grad.flatten() for p in probe.parameters()])
ratio = (g2.norm() / g1.norm()).item()
print(f"||grad|| setelah backward pertama {g1.norm():.3f} | setelah backward kedua (batch sama) "
      f"{g2.norm():.3f} | rasio {ratio:.4f}")
assert abs(ratio - 2.0) < 1e-3
del probe, g1, g2

||grad|| setelah backward pertama 2.537 | setelah backward kedua (batch sama) 5.075 | rasio 2.0000


Rasio 2,000: `backward()` **menambahkan** gradien baru ke `.grad`, bukan menggantinya. Tanpa `zero_grad`, setiap `optimizer.step()` memakai jumlah gradien dari semua batch sebelumnya. Ini berbeda dari *gradient accumulation* yang disengaja, yang menjumlahkan beberapa batch lalu memanggil `step()` dan `zero_grad()` sekali.

Tiga masalah lainnya diperbaiki di fungsi berikut:

- `log_softmax` dan `softmax` dihitung eksplisit pada sumbu vocab (`dim=-1`);
- logits diratakan menjadi satu baris per token, lalu `reduction="batchmean"` membaginya dengan jumlah token, sehingga hasilnya KL rata-rata per token;
- hasil dikalikan **T²**. Temperatur T = 2 melunakkan distribusi teacher sehingga token peringkat kedua dan ketiga ikut membawa informasi (*dark knowledge*). Pelunakan itu juga mengecilkan gradien kira-kira sebesar 1/T², dan faktor T² mengembalikan skalanya.

In [16]:
def distillation_loss(student_logits, teacher_logits, T=2.0):
    """KL(teacher || student) per token pada temperatur T, dikali T^2 agar skala gradien sebanding."""
    vocab = student_logits.size(-1)
    log_p_student = F.log_softmax(student_logits.float().reshape(-1, vocab) / T, dim=-1)
    p_teacher = F.softmax(teacher_logits.float().reshape(-1, vocab) / T, dim=-1)
    return F.kl_div(log_p_student, p_teacher, reduction="batchmean") * T ** 2


z = torch.randn(2, 5, 11, device=DEVICE)
same = distillation_loss(z, z).item()
diff = distillation_loss(z, torch.randn_like(z)).item()
print(f"KD loss untuk logits identik: {same:.1e} | untuk logits berbeda: {diff:.4f}")
assert abs(same) < 1e-5 and diff > 0

KD loss untuk logits identik: 3.3e-08 | untuk logits berbeda: 0.6696


### Under the hood: pangkas 9 dari 12 layer, lalu sembuhkan

Sekarang structured pruning paling ekstrem: student hanya menyimpan 3 blok transformer pertama GPT-2, dan 9 blok sisanya dibuang. Embedding dan ketiga blok itu disalin dari teacher, jadi student mulai dari bobot pretrained, bukan acak.

Dua cara penyembuhan dibandingkan dengan data, jumlah step, learning rate, dan seed yang sama:

- **CE**: fine-tune biasa dengan label keras (token berikutnya yang benar);
- **KD**: distilasi, student meniru distribusi teacher lewat `distillation_loss` di atas.

Teacher adalah GPT-2 utuh yang dibekukan (`requires_grad=False`, mode eval) di GPU, sehingga hanya student yang menyimpan gradien dan state optimizer.

In [17]:
restore()
teacher = gpt2.eval()
teacher.requires_grad_(False)
student = copy.deepcopy(teacher)
student.transformer.h = student.transformer.h[:3]          # structured pruning: 12 -> 3 blok
student.config.n_layer = 3
student.requires_grad_(True)
assert student.lm_head.weight is student.transformer.wte.weight and teacher.config.n_layer == 12
sd_student = {k: v.detach().to("cpu", copy=True) for k, v in student.state_dict().items()}
sliced_loss = eval_loss(student)
print(f"teacher 12 layer: val {BASE_LOSS:.3f} ({N_PARAMS / 1e6:.0f} jt parameter) | "
      f"student 3 layer: val {sliced_loss:.3f} ({sum(p.numel() for p in student.parameters()) / 1e6:.0f} jt)")


def heal(mode, steps, bs, lr=1e-4, T=2.0, seed=0):
    student.load_state_dict(sd_student)
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(student.parameters(), lr=lr)
    student.train()
    losses = []
    for s in range(steps):
        start = (s * bs) % (len(train_ids) - bs + 1)
        x = train_ids[start:start + bs]
        if mode == "CE":
            loss = student(input_ids=x, labels=x, use_cache=False).loss
        else:
            with torch.no_grad():
                t_logits = teacher(input_ids=x, use_cache=False).logits
            loss = distillation_loss(student(input_ids=x, use_cache=False).logits, t_logits, T)
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
        losses.append(loss.detach())
    assert torch.isfinite(torch.stack(losses)).all()
    del opt
    return eval_loss(student)


steps, bs = PROFILE["kd_steps"], PROFILE["kd_batch"]
healed = {}
for mode, tag in (("CE", "fine-tune CE (label keras)"), ("KD", "distilasi KD dari teacher, T=2")):
    t0 = time.perf_counter()
    healed[mode] = heal(mode, steps, bs)
    sync()
    print(f"{tag:<31}: val {sliced_loss:.3f} -> {healed[mode]:.3f} ({steps} step x batch {bs}, "
          f"{time.perf_counter() - t0:.1f} s)")
print(f"selisih KD - CE: {healed['KD'] - healed['CE']:+.3f} nats (negatif = KD lebih baik)")
if DEVICE.type == "cuda":
    print(f"puncak memori GPU sepanjang notebook: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

del student, sd_student
teacher.requires_grad_(True)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

teacher 12 layer: val 4.109 (124 jt parameter) | student 3 layer: val 10.416 (61 jt)
fine-tune CE (label keras)     : val 10.416 -> 6.122 (60 step x batch 4, 4.5 s)
distilasi KD dari teacher, T=2 : val 10.416 -> 6.482 (60 step x batch 4, 7.2 s)
selisih KD - CE: +0.360 nats (negatif = KD lebih baik)
puncak memori GPU sepanjang notebook: 2.15 GB


Baca output ini dalam tiga langkah.

**Seberapa parah luka strukturalnya?** Bandingkan loss student sebelum penyembuhan dengan teacher. Sebagai patokan, menebak seragam dari 50.257 token memberi loss ln(50.257) ≈ 10,8 nats. Run historis mencatat student 3 layer di 10,175, nyaris setara tebakan acak.

**Apakah penyembuhan bekerja?** Kedua cara seharusnya menurunkan loss jauh dari titik itu. Run historis (16 step, batch 2, CPU): CE 10,175 → 7,040 dan KD 10,175 → 6,889, keduanya turun lebih dari 3 nats.

**Mana yang lebih baik?** Di run historis KD unggul 0,15 nats. Profil GPU revisi ini memakai anggaran berbeda (60 step, batch 4), jadi periksa apakah KD juga unggul di run Anda. Dengan satu seed dan anggaran sekecil ini, selisih sepersepuluh nats adalah petunjuk, bukan bukti; kesimpulan yang kuat butuh beberapa seed dan training lebih panjang. Prinsipnya sendiri sudah dipakai di skala besar: DistilGPT-2 adalah student 6 layer yang didistilasi dari GPT-2 dengan data yang jauh lebih banyak.

## Ringkasan

Pada GPT-2, keberhasilan pruning lebih ditentukan oleh *apa* dan *kapan* dipangkas daripada oleh fungsi mana yang dipanggil.

- **Sasaran.** GPT-2 di `transformers` hanya punya satu `nn.Linear` (`lm_head`, yang tied dengan embedding input) dan 48 `Conv1D`. Loop `isinstance(nn.Linear)` hanya memangkas 9,3% parameter, dan semuanya dari embedding.
- **Seberapa jauh.** Magnitude pruning pada `Conv1D` membentuk kurva datar lalu tebing: porsi kecil bobot bisa dibuang hampir gratis, tetapi kerusakan melonjak setelah titik patah.
- **Cara kerja.** `torch.nn.utils.prune` memakai reparametrisasi (`weight_orig * weight_mask`) agar gradien tetap sampai ke parameter yang dipegang optimizer. Reimplementasi yang membuat `nn.Parameter` baru di setiap forward membekukan training.
- **Struktur.** Pada `Conv1D`, neuron ada di `dim=1`, dan menghapus neuron sepenuhnya berarti ikut menghapus biasnya. Structured pruning lebih mahal dalam loss, tetapi hanya ia yang bisa diubah menjadi matriks yang lebih kecil. Mask dan nol tersebar tidak mempercepat perkalian padat.
- **Urutan.** `remove()` dipanggil sekali di akhir. Memanggilnya tiap ronde membuat pruning iteratif macet, dan fine-tune setelah `remove()` menumbuhkan kembali bobot yang dibuang.
- **Kompresi.** Nol tidak mengecilkan file; CSR baru hemat di atas sekitar 67% sparsity; dan quantization dinamis `{nn.Linear}` pada GPT-2 justru membesarkan checkpoint sekaligus merusak loss.
- **Distilasi.** Student 3 layer hasil pemangkasan brutal bisa disembuhkan, dan distilasi dari teacher adalah kandidat penyembuh yang kuat. Satu run kecil belum cukup untuk memastikan keunggulannya atas fine-tune biasa.

| Listing buku | Yang terjadi pada GPT-2 | Versi yang bekerja di notebook ini |
|---|---|---|
| `model = ...` + loop `nn.Linear` | error `Ellipsis`; setelah diisi, hanya `lm_head`/`wte` yang terpangkas | loop atas `Conv1D` |
| `def prune.l1_unstructured(...)` | `SyntaxError`; setelah dinamai ulang, training membeku | `torch.nn.utils.prune` |
| `ln_structured(..., dim=0)` | memangkas fitur input, bukan neuron | `dim=1` + mask bias, lalu bangun ulang layer |
| iterasi dengan `remove()` tiap ronde | macet di 10% | mask menumpuk, `remove()` sekali di akhir |
| `criterion(outputs, targets)` | `TypeError`; label tidak digeser | `model(input_ids=x, labels=x).loss` |
| `torch.cuda.memory_allocated()` sebagai ukuran | `0` dan `0` untuk model CPU | ukuran `state_dict` terserialisasi |
| loop distilasi | error `Ellipsis`; tanpa `zero_grad`, `dim`, `batchmean`, T² | `distillation_loss` terkoreksi + `zero_grad` |

Bab berikutnya di buku membahas **quantization** secara khusus, termasuk cara meng-quantize lapisan yang di bab ini terlewat oleh `{nn.Linear}`.